# LC6 — Power system data sources and acquisition (self-paced, ~45 min)

Real analysis starts with real data. This notebook takes you from "where does power system data live?" to a cached, provenance-stamped fetch from the ENTSO-E Transparency Platform — the pattern your Lab 5 pipeline builds on. Material from this notebook appears in **Quiz 2**.

**You need:** your ENTSO-E API token (requested during course setup). No token yet? Every cell still runs — the fetch falls back to a bundled sample so you can complete the notebook offline, but do the real fetch before Lab 5.

## 1. Where the data lives — a five-minute tour

- **[ENTSO-E Transparency Platform](https://transparency.entsoe.eu)** — the European hub: load, generation per type, day-ahead prices, cross-border flows. Web GUI for browsing, REST API for programs. This is today's workhorse.
- **TSO data pools** — [Svenska kraftnät (Mimer)](https://mimer.svk.se), [Fingrid open data](https://data.fingrid.fi), Statnett. Often richer than ENTSO-E for their own area.
- **[Open Infrastructure Map](https://openinframap.org)** — the grid itself, from OpenStreetMap data.
- **System state files** — the CGMES snapshots you meet in Lecturecise 8: not measurements over time but the *network* at an instant.

Three questions to ask of any source, always: who publishes it, what exactly does a value mean (metered? forecast? aggregated how?), and what are you allowed to do with it.

In [ ]:
# Install exactly what this notebook uses (transitive deps come along).
%pip install entsoe-py pandas pyarrow matplotlib --quiet

In [ ]:
from pathlib import Path
# Guard: this notebook expects to run from the notebooks/ folder of a clone of
# the course repository — the datasets live one level up in ../data/.
# Failing here, early and clearly, beats a confusing FileNotFoundError later.
assert Path("../data").exists(), (
    "Course data folder not found. Clone KTH-EG2140/course-material and open "
    "this notebook from its notebooks/ folder.")

## 2. Your token

The API wants a token with every request. Never hardcode it in a notebook you will commit — read it from a file or an environment variable (this is the pattern for every secret, all course long):

In [ ]:
import os
from pathlib import Path

# Option A: a file next to this notebook (add it to .gitignore!)
# Option B: an environment variable ENTSOE_TOKEN
token_file = Path("entsoe_token.txt")
TOKEN = token_file.read_text().strip() if token_file.exists() else os.environ.get("ENTSOE_TOKEN")
print("token found" if TOKEN else "no token — the notebook will use the offline fallback")

## 3. Fetch with a cache

Two rules of API citizenship: **never fetch the same thing twice** (a cache), and **record where every dataset came from** (provenance). Both in one small function:

In the lecture the same fetch was one raw REST call: `documentType=A65` (actual total load) against a zone's **EIC code** — the example queried SE4 by its code `10Y1001A1001A47J`. The `entsoe-py` client below wraps exactly that request: when you write `"SE_3"`, the library looks up the zone's EIC code (`10Y1001A1001A46L`) and fills in the document type for you. Same API, same parameters — the library just does the bookkeeping.


In [ ]:
import json
from datetime import datetime, timezone

import pandas as pd

CACHE = Path("data_cache"); CACHE.mkdir(exist_ok=True)

def cached_load(zone: str, start: str, end: str) -> pd.Series:
    """Hourly actual load for an ENTSO-E zone, cached in Parquet with a provenance sidecar."""
    key = f"load_{zone}_{start}_{end}"
    pq, meta = CACHE / f"{key}.parquet", CACHE / f"{key}.json"
    if pq.exists():
        print(f"cache hit: {pq.name}")
        return pd.read_parquet(pq)["load_mw"]
    from entsoe import EntsoePandasClient          # imported here: offline users never need it
    client = EntsoePandasClient(api_key=TOKEN)
    s = client.query_load(zone, start=pd.Timestamp(start, tz="Europe/Stockholm"),
                          end=pd.Timestamp(end, tz="Europe/Stockholm"))
    s = s.iloc[:, 0].resample("1h").mean().rename("load_mw")   # 15-min zones -> hourly
    s.to_frame().to_parquet(pq)
    meta.write_text(json.dumps({
        "source": "ENTSO-E Transparency, query_load", "zone": zone,
        "period": [start, end], "retrieved_utc": datetime.now(timezone.utc).isoformat(),
        "unit": "MW, hourly mean of source resolution"}, indent=2))
    print(f"fetched and cached: {pq.name}")
    return s

In [ ]:
# The real thing (needs the token) — with an honest offline fallback.
try:
    if not TOKEN:
        raise RuntimeError("no token")
    load = cached_load("SE_3", "2025-06-01", "2025-07-01")
    SOURCE = "ENTSO-E (live or cache)"
except Exception as e:
    print(f"[offline fallback: {type(e).__name__}: {e}]")
    # course sample — SAME SHAPE as the real answer, clearly labelled, so the
    # rest of the notebook works on the train. Do the real fetch before Lab 5.
    df = pd.read_parquet("../data/svedala-year/svedala_hourly.parquet")
    load = df["ZON_MITT"].loc["2025-06-01":"2025-06-30"].rename("load_mw")
    SOURCE = "COURSE SAMPLE (offline fallback) — not ENTSO-E"
print(SOURCE, "|", len(load), "hours |", f"mean {load.mean():.0f} MW")

Run the fetch cell **twice** and watch the second run hit the cache — that is the whole point. Then look inside `data_cache/`: the `.json` sidecar is the dataset's birth certificate. When someone asks "where is this number from?" in your project, that file is the answer.

In [ ]:
ax = load.plot(figsize=(10, 3), title=f"Hourly load — {SOURCE}")
ax.set_ylabel("MW");

## 4. Rate limits and manners

The Transparency API allows a limited number of requests per minute and will lock you out if you hammer it (HTTP 429). Consequences for how you write code: fetch **big chunks rarely** (a month, not 720 single hours), cache everything, and put a `time.sleep(1)` between calls in any loop. Your Lab 5 pipeline is expected to survive a rerun without re-fetching a single byte.

## 5. Self-check

All three should pass — offline or online:

In [ ]:
assert len(load) >= 24 * 28, "expected at least a month of hourly data"
assert load.notna().mean() > 0.95, "too many gaps"
assert 500 < load.mean() < 50000, "mean load outside any plausible zone range"
print("ALL OK —", SOURCE)
if "fallback" in SOURCE:
    print("…but do the real ENTSO-E fetch before Lab 5. The quiz assumes you have.")